# circuit_lm — Train + Evaluate + Trace (WandB + Kaggle)

Full pipeline: train a PDA circuit + neural corrector hybrid on your data,
then trace through predictions to see *why* the model made each choice.

**Runtime**: GPU recommended for corrector training (Colab T4/V100). Circuit training is CPU-fast.
**Weights & Biases**: wandb logs metrics automatically if wandb token is in Kaggle secrets.

**Estimated time**: ~5-10 min for full pipeline on 100K examples.

## 1. Setup — clone repo + install deps + wandb login

In [ ]:
# Clone repo
!git clone https://github.com/toxzak-svg/circuit_lm.git 2>&1 | tail -3
%cd circuit_lm

# Install deps
!pip install -q ortools pandas wandb 2>&1 | tail -3

# Load Kaggle + wandb secrets from Kaggle Secrets (Colab)
import os, subprocess

try:
    from google.colab import userdata
    
    # Kaggle credentials — set KAGGLE_USERNAME and KAGGLE_API_KEY in Kaggle Secrets
    kaggle_user = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_API_KEY')
    if kaggle_user and kaggle_key:
        os.environ['KAGGLE_USERNAME'] = kaggle_user
        os.environ['KAGGLE_API_KEY'] = kaggle_key
        print(f"Kaggle creds loaded for user: {kaggle_user}")
    else:
        print("Kaggle secrets not found — skipping")
    
    # WandB token — set WANDB_API_KEY in Kaggle Secrets
    wandb_token = userdata.get('WANDB_API_KEY')
    if wandb_token:
        os.environ['WANDB_API_KEY'] = wandb_token
        # Verify wandb login
        result = subprocess.run(['wandb', 'who'], capture_output=True, text=True)
        print(f"WandB logged in as: {result.stdout.strip()}")
    else:
        print("WANDB_API_KEY not found in secrets — wandb logging disabled")
        
except Exception as e:
    print(f"Colab secrets not available: {e} — proceeding without external creds")

## 2. Upload your training data

Upload a `.txt` file. For chat data use `User:` / `Assistant:` prefix format.

In [ ]:
from google.colab import files

print("Upload your training data (.txt file):")
uploaded = files.upload()
data_path = list(uploaded.keys())[0]
print(f"Uploaded: {data_path} ({(len(uploaded[data_path])/1e6):.1f} MB)")

## 3. Train Circuit (FSM or PDA)

In [ ]:
import subprocess, json, time, os

# Config — edit these
AUTOMATON = "pda"          # "fsm", "pda", or "ppm"
VOCAB_SIZE = 1024
STATE_BITS = 5             # 2^5 = 32 states
STACK_DEPTH = 6            # PDA stack depth
BPE_MERGES = 512           # BPE vocabulary size
CIRCUIT_OUT = "circuit.json"
RUN_NAME = f"circuit_{AUTOMATON}_{VOCAB_SIZE}vocab"  # for wandb

# Init wandb for circuit training
if os.environ.get('WANDB_API_KEY'):
    import wandb
    wandb.init(name=RUN_NAME, project="circuit-lm", entity="zacharymaronek",
               settings=wandb.Settings(_stats_window_minutes=1))
    wandb.config.update({
        "step": "circuit",
        "automaton": AUTOMATON,
        "vocab_size": VOCAB_SIZE,
        "state_bits": STATE_BITS,
        "stack_depth": STACK_DEPTH,
        "bpe_merges": BPE_MERGES,
    })

cmd = [
    "python", "-m", "circuit_lm.cli", "train",
    "--data", data_path,
    "--out", CIRCUIT_OUT,
    "--automaton", AUTOMATON,
    "--vocab_size", str(VOCAB_SIZE),
    "--state_bits", str(STATE_BITS),
    "--stack_depth", str(STACK_DEPTH),
    "--tokenizer", "bpe",
    "--bpe_merges", str(BPE_MERGES),
    "--steps", "60",
]
print(f"Running: {' '.join(cmd)}")
t0 = time.time()
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Training failed")
print(f"Circuit trained in {time.time()-t0:.1f}s")

## 4. Train Hybrid Corrector (Neural Net + WandB)

In [ ]:
import subprocess, shutil, time, os

CORRECTOR_OUT = "corrector.pt"

# Init wandb for corrector training
if os.environ.get('WANDB_API_KEY'):
    import wandb
    wandb.init(name=f"corrector_{AUTOMATON}", project="circuit-lm", entity="zacharymaronek",
               settings=wandb.Settings(_stats_window_minutes=1))
    wandb.config.update({
        "step": "corrector",
        "automaton": AUTOMATON,
        "epochs": 5,
        "batch_size": 128,
        "max_examples": 100000,
        "embed_dim": 256,
        "hidden_dim": 512,
        "num_layers": 3,
        "streaming": True,
    })

# Run corrector training
cmd = [
    "python", "scripts/train_bpe_hybrid.py",
    "--data", data_path,
    "--circuit-out", CIRCUIT_OUT,
    "--corrector-out", CORRECTOR_OUT,
    "--automaton", AUTOMATON,
    "--vocab-size", str(VOCAB_SIZE),
    "--bpe-merges", str(BPE_MERGES),
    "--state-bits", str(STATE_BITS),
    "--stack-depth", str(STACK_DEPTH),
    "--epochs", "5",
    "--batch-size", "128",
    "--max-examples", "100000",
    "--embed-dim", "256",
    "--hidden-dim", "512",
    "--num-layers", "3",
    "--streaming",
]

t0 = time.time()
result = subprocess.run(cmd, capture_output=True, text=True)

# Log final metrics to wandb
if os.environ.get('WANDB_API_KEY'):
    import wandb
    # Parse output for accuracy
    for line in result.stdout.split('\n'):
        if 'accuracy=' in line:
            # Extract accuracy value
            acc_str = line.split('accuracy=')[-1].strip().rstrip('%')
            try:
                wandb.log({"final_accuracy": float(acc_str)})
            except:
                pass
    wandb.finish()

print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Corrector training failed")
print(f"\nCorrector trained in {time.time()-t0:.1f}s")

## 5. Trace — See Why Each Token Was Predicted

In [ ]:
# Trace through a prompt and see state + stack + top-k predictions at each step
PROMPT = "User: hello\nAssistant:"  # edit this
TOP_K = 5

cmd = [
    "python", "-m", "circuit_lm.cli", "trace",
    "--model", CIRCUIT_OUT,
    "--prompt", PROMPT,
    "--top_k", str(TOP_K),
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

**What you're seeing:**
- **step**: position in token sequence
- **token**: actual next token (ground truth)
- **state**: FSM/PDA state before seeing this token
- **stack_top**: PDA stack top (-1 = empty)
- **top_k**: model's ranked predictions — if gold token isn't here, model was wrong

Analyzing where the model fails = what the corrector needs to fix.

## 6. Evaluate Accuracy

In [ ]:
import random, pathlib

all_text = pathlib.Path(data_path).read_text()
lines = [l for l in all_text.split('\n') if l.strip()]
random.seed(42)
random.shuffle(lines)
split = int(len(lines) * 0.9)
eval_text = '\n'.join(lines[split:])
eval_path = "eval_data.txt"
pathlib.Path(eval_path).write_text(eval_text)

cmd = [
    "python", "-m", "circuit_lm.cli", "eval",
    "--model", CIRCUIT_OUT,
    "--data", eval_path,
    "--per_token",
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

## 7. Download Trained Models

In [ ]:
from google.colab import files

files.download(CIRCUIT_OUT)
files.download(CORRECTOR_OUT)
print(f"\ncircuit.json + corrector.pt saved")
print(f"\nTo run locally:")
print(f"  py -3.12 -m circuit_lm.cli chat --model circuit.json --corrector corrector.pt")
print(f"  py -3.12 -m circuit_lm.cli trace --model circuit.json --prompt 'User: hi' --top_k 5")